In [1]:
# ── 셀 1 │ 설정 ──
import os, sys, json, time, random
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import add_self_loops, softmax
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

NEW  = r'C:\Users\iisl\Desktop\0_Molecualar_Informatics\2026_final\2026_new' + '\\'
OUT  = NEW + r'03_dedup\\'
HERG_POOL = NEW + r'data\processed\herg_karim\herg_karim_pool.csv'
CLIN = NEW + r'data\processed\clinical\clinical_pool_endpoints.csv'
os.makedirs(OUT, exist_ok=True)
assert os.path.exists(HERG_POOL) and os.path.exists(CLIN)

sys.path.insert(0, NEW + '01_pre-train')      # featurizer.py 위치
import featurizer as FZ

DEV  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
PRE_LR, PRE_EPOCHS, PRE_PAT, PRE_BS = 1e-3, 200, 20, 64

print(DEV, '| atom', FZ.num_atom_features(), '| bond', FZ.num_bond_features())

C:\Users\iisl\anaconda3\envs\esu_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda | atom 39 | bond 10


In [2]:
# ── 셀 2 │ GAT_SI ──
class GH(nn.Module):
    def __init__(s, ic, oc, d=0.3):
        super().__init__()
        s.W = nn.Parameter(torch.empty(ic, oc))
        s.a = nn.Parameter(torch.empty(2 * oc, 1))
        nn.init.xavier_uniform_(s.W); nn.init.xavier_uniform_(s.a)
        s.lr = nn.LeakyReLU(0.2); s.d = d
    def forward(s, x, ei):
        ei, _ = add_self_loops(ei, num_nodes=x.size(0))
        h = x @ s.W
        row, col = ei
        e = s.lr((torch.cat([h[row], h[col]], -1) @ s.a).squeeze(-1))
        e = F.dropout(e, s.d, training=s.training)
        al = softmax(e, row)
        o = torch.zeros_like(h)
        o.index_add_(0, row, h[col] * al.unsqueeze(-1))
        return F.elu(o)

class GL(nn.Module):
    def __init__(s, ic, oc, h, d=0.3):
        super().__init__()
        s.h, s.d = h, d
        s.att = nn.ModuleList([GH(ic, oc, d) for _ in range(h)])
    def forward(s, x, ei):
        if s.h == 1:
            z = torch.mean(torch.stack([a(x, ei) for a in s.att], 0), 0)
        else:
            z = torch.cat([a(x, ei) for a in s.att], -1)
        return F.dropout(z, s.d, training=s.training)

class AttentionPool(nn.Module):
    def __init__(s, ch, d=0.3):
        super().__init__()
        s.gate = nn.Sequential(nn.Linear(ch, ch), nn.ReLU(),
                               nn.Dropout(d), nn.Linear(ch, 1))
    def forward(s, x, batch):
        al = softmax(s.gate(x).squeeze(-1), batch)
        n = int(batch.max()) + 1
        out = torch.zeros(n, x.size(1), device=x.device, dtype=x.dtype)
        out.index_add_(0, batch, x * al.unsqueeze(-1))
        return out

class GAT_SI(nn.Module):
    def __init__(s, ic, hc=32, h=8, d=0.3):
        super().__init__()
        s.l1 = GL(ic, hc, h, d)
        s.l2 = GL(hc * h, hc, h, d)
        s.l3 = GL(hc * h, hc, 1, d)
        s.pool = AttentionPool(hc, d)
        s.clf = nn.Sequential(nn.Linear(hc, 64), nn.ReLU(),
                              nn.Dropout(d), nn.Linear(64, 1))
    def backbone_parameters(s):
        return (list(s.l1.parameters()) + list(s.l2.parameters())
                + list(s.l3.parameters()) + list(s.pool.parameters()))
    def forward(s, x, ei, b):
        x = F.elu(s.l1(x, ei)); x = F.elu(s.l2(x, ei)); x = s.l3(x, ei)
        return s.clf(s.pool(x, b)).squeeze(-1)

print('params', sum(p.numel() for p in GAT_SI(FZ.num_atom_features()).parameters()))

params 88066


In [3]:
# ── 셀 3 │ hERG 그래프 ──
def key14(smi):
    m = Chem.MolFromSmiles(smi)
    return None if m is None else Chem.MolToInchiKey(m)[:14]

def _prep(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try: Chem.SanitizeMol(m)
    except Exception: return None
    try: Chem.AssignStereochemistry(m, cleanIt=True, force=True)
    except Exception: pass
    return m

def make_graph(smi, y=None):
    mol = _prep(smi)
    if mol is None: return None
    try: x = np.stack([FZ.atom_features(a) for a in mol.GetAtoms()])
    except Exception: return None
    src, dst, ea = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = FZ.bond_features(b)
        src += [i, j]; dst += [j, i]; ea += [bf, bf]
    if not src: return None
    g = Data(x=torch.tensor(np.asarray(x, np.float32)),
             edge_index=torch.tensor([src, dst], dtype=torch.long),
             edge_attr=torch.tensor(np.asarray(ea, np.float32)))
    if y is not None:
        g.y = torch.tensor([float(y)], dtype=torch.float)
    return g

CACHE = OUT + 'graphs_herg_pool11037.pt'
if os.path.exists(CACHE):
    blob = torch.load(CACHE, weights_only=False)
    Hg, Hm = blob['graphs'], blob['meta']
    print('캐시 로드', len(Hg))
else:
    # 전처리 노트북이 확정한 풀을 그대로 쓴다. raw .tab 에서 다시 만들면
    # 라벨 충돌 83종 처리 정책이 달라져 공유 화합물이 209 가 아니라 216 이 된다.
    herg = pd.read_csv(HERG_POOL)
    herg = herg.rename(columns={'Y': 'y'})
    print('hERG pool', len(herg), dict(herg['y'].value_counts()))
    assert len(herg) == 11037, len(herg)
    Hg, rec = [], []
    for _, r in herg.iterrows():
        g = make_graph(r['canon_smiles'], r['y'])
        if g is None: continue
        Hg.append(g); rec.append({'key14': r['key14'],
                                  'smiles': r['canon_smiles'],
                                  'y': int(r['y'])})
    Hm = pd.DataFrame(rec)
    torch.save({'graphs': Hg, 'meta': Hm}, CACHE)
    print('생성', len(Hg))

print(len(Hg), dict(Hm['y'].value_counts()))

hERG pool 11037 {0: 5610, 1: 5427}
생성 11037
11037 {0: 5610, 1: 5427}


In [4]:
# ── 셀 4 │ 분할 먼저 고정, train/val 에서만 제거 ──
idx   = np.arange(len(Hg))
y_all = Hm['y'].values.astype(int)

tr_idx, te_idx = train_test_split(idx, test_size=0.10,
                                  random_state=SEED, stratify=y_all)
tr_idx, va_idx = train_test_split(tr_idx, test_size=1/9,
                                  random_state=SEED, stratify=y_all[tr_idx])

clin = pd.read_csv(CLIN, encoding='utf-8-sig')
clin_y = dict(zip(clin['key14'], clin['y'].astype(int)))
ov = Hm['key14'].isin(clin_y).values
print(f'공유 {ov.sum()}  (train {ov[tr_idx].sum()} / '
      f'val {ov[va_idx].sum()} / test {ov[te_idx].sum()})')

# Table 4 교차표
sh = Hm.loc[ov, ['key14','y']].copy()
sh['clinical'] = sh['key14'].map(clin_y)
xt = pd.crosstab(sh['y'], sh['clinical'],
                 rownames=['hERG'], colnames=['clinical'], margins=True)
print(xt)
conc = int(xt.loc[0,0] + xt.loc[1,1]); n = int(xt.loc['All','All'])
print(f'concordant {conc}/{n} = {conc/n:.1%} | discordant {n-conc} = {(n-conc)/n:.1%}')
print(f'hERG-inactive & clinically toxic {int(xt.loc[0,1])} vs '
      f'hERG-active & non-toxic {int(xt.loc[1,0])}')
xt.to_csv(OUT + 'table4_crosstab.csv')

ARMS = {'keep': (tr_idx,              va_idx),
        'drop': (tr_idx[~ov[tr_idx]], va_idx[~ov[va_idx]])}
for k, (t, v) in ARMS.items():
    print(f'{k:5s} train {len(t):6d}  val {len(v):5d}  test {len(te_idx):5d}')

json.dump({'train': tr_idx.tolist(), 'val': va_idx.tolist(),
           'test': te_idx.tolist(), 'overlap': ov.tolist(), 'seed': SEED},
          open(OUT + 'split_idx.json', 'w'))

공유 209  (train 163 / val 20 / test 26)
clinical   0    1  All
hERG                  
0         22   98  120
1         14   75   89
All       36  173  209
concordant 97/209 = 46.4% | discordant 112 = 53.6%
hERG-inactive & clinically toxic 98 vs hERG-active & non-toxic 14
keep  train   8829  val  1104  test  1104
drop  train   8666  val  1084  test  1104


In [5]:
# ── 셀 5 │ fit / metrics ──
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def metrics(y, p, thr=0.5):
    yh = (np.asarray(p) >= thr).astype(int); y = np.asarray(y).astype(int)
    return dict(accuracy=accuracy_score(y, yh),
                auroc=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
                aucpr=average_precision_score(y, p) if len(set(y)) > 1 else np.nan,
                f1=f1_score(y, yh, zero_division=0),
                precision=precision_score(y, yh, zero_division=0),
                recall=recall_score(y, yh, zero_division=0))

@torch.no_grad()
def predict(model, loader):
    model.eval(); ps, ys = [], []
    for b in loader:
        b = b.to(DEV)
        ps += torch.sigmoid(model(b.x, b.edge_index, b.batch)).cpu().tolist()
        ys += b.y.view(-1).cpu().tolist()
    return np.array(ys), np.array(ps)

def fit(model, tr_ds, va_ds, opt, epochs, patience, bsz):
    tr = DataLoader(tr_ds, batch_size=bsz, shuffle=True)
    vl = DataLoader(va_ds, batch_size=256)
    npos = sum(int(g.y.item()) for g in tr_ds); nneg = len(tr_ds) - npos
    pw = torch.tensor([max(nneg,1)/max(npos,1)], dtype=torch.float).to(DEV)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    best, state, w = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        for b in tr:
            b = b.to(DEV); opt.zero_grad()
            crit(model(b.x, b.edge_index, b.batch),
                 b.y.view(-1).float()).backward()
            opt.step()
        ys, ps = predict(model, vl)
        a = roc_auc_score(ys, ps) if len(set(ys.astype(int))) > 1 else 0.5
        if a > best:
            best, w = a, 0
            state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            w += 1
            if w >= patience: break
    if state:
        model.load_state_dict({k: v.to(DEV) for k, v in state.items()})
    return model, best

In [6]:
# ── 셀 6 │ 사전학습 ──
IC = FZ.num_atom_features()
te_ds = [Hg[i] for i in te_idx]
rows = {}

for arm, (t, v) in ARMS.items():
    t0 = time.time(); set_seed(SEED)
    m = GAT_SI(IC).to(DEV)
    opt = torch.optim.Adam(m.parameters(), lr=PRE_LR, weight_decay=5e-4)
    m, bestv = fit(m, [Hg[i] for i in t], [Hg[i] for i in v],
                   opt, PRE_EPOCHS, PRE_PAT, PRE_BS)
    ys, ps = predict(m, DataLoader(te_ds, batch_size=256))
    r = metrics(ys, ps); r['val_auroc'] = bestv
    rows[arm] = r
    torch.save(m.state_dict(), OUT + f'pretrain_{arm}.pt')
    print(f'[{arm}] {time.time()-t0:.0f}s  ' +
          '  '.join(f'{k} {v:.4f}' for k, v in r.items()), flush=True)

src = pd.DataFrame(rows).T[['accuracy','auroc','aucpr','f1',
                            'precision','recall']].round(4)
src.to_csv(OUT + 'table5_source.csv')
src

[keep] 750s  accuracy 0.7301  auroc 0.8076  aucpr 0.7972  f1 0.7444  precision 0.6966  recall 0.7993  val_auroc 0.8074
[drop] 928s  accuracy 0.7328  auroc 0.8040  aucpr 0.7939  f1 0.7166  precision 0.7490  recall 0.6869  val_auroc 0.8107


,accuracy,auroc,aucpr,f1,precision,recall
keep,0.7301,0.8076,0.7972,0.7444,0.6966,0.7993
drop,0.7328,0.8040,0.7939,0.7166,0.7490,0.6869


In [7]:
# ── 저장 ──
import json, os
os.makedirs(OUT, exist_ok=True)

ckpt = {
    'state_dict': {k: v.cpu() for k, v in m.state_dict().items()},
    'arch': dict(cls='GAT_SI', ic=IC, hc=32, heads=8, dropout=0.3),
    'train': dict(loss='wbce', lr=PRE_LR, epochs=PRE_EPOCHS,
                  patience=PRE_PAT, batch=PRE_BS, seed=SEED),
    'split': dict(train=ARMS[arm][0].tolist(),
                  val=ARMS[arm][1].tolist(),
                  test=te_idx.tolist()),
    'arm': arm,
    'n_overlap_removed': int(ov[tr_idx].sum() + ov[va_idx].sum()) if arm == 'drop' else 0,
}
torch.save(ckpt, OUT + f'pretrain_{arm}_full.pt')

# 미세조정 노트북이 바로 load_state_dict 할 수 있는 가벼운 버전도 같이
torch.save(ckpt['state_dict'], OUT + f'pretrain_{arm}.pt')
print('저장', OUT + f'pretrain_{arm}.pt')

저장 C:\Users\iisl\Desktop\0_Molecualar_Informatics\2026_final\2026_new\03_dedup\\pretrain_drop.pt


In [8]:
# ── 매니페스트 ──
man = {'arms': {}, 'test_idx_n': len(te_idx), 'seed': SEED,
       'herg_n': len(Hg), 'n_shared': int(ov.sum())}
for a in ['keep', 'drop']:
    t, v = ARMS[a]
    man['arms'][a] = {'n_train': int(len(t)), 'n_val': int(len(v)),
                      'ckpt': f'pretrain_{a}.pt',
                      'metrics': {k: float(x) for k, x in rows[a].items()}}
json.dump(man, open(OUT + 'pretrain_manifest.json', 'w',
                    encoding='utf-8'), indent=2, ensure_ascii=False)
print(json.dumps(man, indent=2))

{
  "arms": {
    "keep": {
      "n_train": 8829,
      "n_val": 1104,
      "ckpt": "pretrain_keep.pt",
      "metrics": {
        "accuracy": 0.730072463768116,
        "auroc": 0.8075555686865403,
        "aucpr": 0.7971998214373517,
        "f1": 0.7444253859348199,
        "precision": 0.6966292134831461,
        "recall": 0.7992633517495396,
        "val_auroc": 0.8074045623606885
      }
    },
    "drop": {
      "n_train": 8666,
      "n_val": 1084,
      "ckpt": "pretrain_drop.pt",
      "metrics": {
        "accuracy": 0.7327898550724637,
        "auroc": 0.8039511133433785,
        "aucpr": 0.793936936187662,
        "f1": 0.7166186359269933,
        "precision": 0.748995983935743,
        "recall": 0.6869244935543278,
        "val_auroc": 0.810738108777151
      }
    }
  },
  "test_idx_n": 1104,
  "seed": 42,
  "herg_n": 11037,
  "n_shared": 209
}


In [9]:
for a in ['keep', 'drop']:
    sd = torch.load(OUT + f'pretrain_{a}.pt', map_location='cpu', weights_only=True)
    t = GAT_SI(IC)
    missing, unexpected = t.load_state_dict(sd, strict=False)
    print(a, '| missing', missing, '| unexpected', unexpected)
    assert not missing and not unexpected

keep | missing [] | unexpected []
drop | missing [] | unexpected []
